# Colab Single-Model Judge Template

Use this notebook in Colab for one model at a time.

What it does:
- clones this repo
- loads `benchmark_dataset/1000_questions_dataset.csv`
- runs one model selected in the config cell
- saves incremental CSV checkpoints and a final output file

For your 4 Colab runs, duplicate the notebook 4 times and change only the `MODEL_ID` cell.

In [ ]:
# =========================
# CONFIG: change only this
# =========================
REPO_URL = 'https://github.com/mmm-byte/Multi_LLM_Medical_AI_Judge_copy.git'
BRANCH = 'main'
MODEL_ID = 'google/gemma-2-2b-it'   # change per notebook
JUDGE_NAME = 'model_1'             # change per notebook
DATASET_PATH = '/content/Multi_LLM_Medical_AI_Judge_copy/benchmark_dataset/1000_questions_dataset.csv'
OUTPUT_DIR = '/content/output'
OUTPUT_CSV = f'{OUTPUT_DIR}/{JUDGE_NAME}_results.csv'
CHECKPOINT_EVERY = 25
MAX_ROWS = 1000
BATCH_SIZE = 1
MAX_NEW_TOKENS = 64
TEMPERATURE = 0.0


In [ ]:
!pip -q install -U transformers accelerate datasets pandas sentencepiece bitsandbytes
!rm -rf /content/Multi_LLM_Medical_AI_Judge_copy
!git clone --depth 1 -b $BRANCH $REPO_URL /content/Multi_LLM_Medical_AI_Judge_copy


In [ ]:
import os, csv, json, math, time
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(DATASET_PATH).head(MAX_ROWS).copy()
df[['id', 'domain', 'question', 'answer', 'expected_class']].head(3)


In [ ]:
use_4bit = torch.cuda.is_available()
quant_cfg = None
if use_4bit:
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    quantization_config=quant_cfg,
)
model.eval()
print('Loaded', MODEL_ID)


In [ ]:
def build_prompt(question: str, answer: str) -> str:
    return (
        'You are a strict medical domain judge evaluating a clinical QA answer. '\n        'Return one line only with the predicted agreement class: '\n        'fully_agree, majority_agree, split, or full_disagree.\n\n'
        f'Question: {question}\n'
        f'Answer: {answer}\n\n'
        'Class:'
    )

def generate_one(question: str, answer: str) -> str:
    prompt = build_prompt(question, answer)
    if getattr(tokenizer, 'chat_template', None):
        messages = [
            {'role': 'system', 'content': 'You are a strict medical domain judge.'},
            {'role': 'user', 'content': prompt},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=TEMPERATURE > 0,
            temperature=max(TEMPERATURE, 1e-5),
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()

def parse_class(raw: str) -> str:
    txt = raw.lower()
    for cls in ['fully_agree', 'majority_agree', 'split', 'full_disagree']:
        if cls in txt:
            return cls
    return 'unparsed'

rows = []
for i, row in df.iterrows():
    raw = generate_one(str(row['question']), str(row['answer']))
    rows.append({
        'id': row['id'],
        'domain': row['domain'],
        'question': row['question'],
        'answer': row['answer'],
        'expected_class': row['expected_class'],
        'model_id': MODEL_ID,
        'judge_name': JUDGE_NAME,
        'raw_output': raw,
        'predicted_class': parse_class(raw),
    })
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
        print(f'Checkpoint saved: {OUTPUT_CSV} ({i+1} rows)')

out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False)
print('Final saved:', OUTPUT_CSV)
out_df.head(5)


In [ ]:
summary = pd.read_csv(OUTPUT_CSV)
print(summary[['domain', 'expected_class', 'predicted_class']].head())
print(summary['predicted_class'].value_counts(dropna=False))
print('Saved to:', OUTPUT_CSV)
